# TrendLens — 06 · Retrieval over Cluster Interpretations (Phase 6)

Stages 10 + 12: CLIP text embeddings of Phase 5 interpretations (same `clip-vit-base-patch32` as the image side) → FAISS flat inner-product index → top-k cluster retrieval → hit@k / MRR against **human-curated** query labels.

> **Integrity:** no real per-post SMPD labels exist locally. Evaluation labels were curated from Phase 5 BLIP captions of the representative images (recorded in `cluster_captions.json`). hit@k measures retrieval agreement with that curation — real, but scoped.

In [1]:
import sys
from pathlib import Path
REPO = Path.cwd()
if not (REPO / "config.py").exists():
    for p in Path.cwd().parents:
        if (p / "config.py").exists():
            REPO = p; break
sys.path.insert(0, str(REPO))

import json, numpy as np, pandas as pd
import config
from src import retrieval

## 1 · Load interpretations + embed with CLIP text encoder

In [2]:
interpretations = json.loads(
    (config.CLUSTER_METADATA_DIR / "cluster_captions.json").read_text())["interpretations"]
model, processor, device = retrieval.load_clip_text()
print("CLIP text on", device)
embs, cluster_ids = retrieval.embed_interpretations(model, processor, interpretations, device=device)
print("embeddings:", embs.shape, "| clusters:", len(cluster_ids))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP text on cpu
embeddings: (29, 512) | clusters: 29


## 2 · Build + persist FAISS index

In [3]:
index = retrieval.build_index(np.asarray(embs))
p = retrieval.save_index(index)
print("index:", index.ntotal, "vectors ->", p.name)

index: 29 vectors -> cluster_index.faiss


## 3 · Hand-written query demo

In [4]:
def show_top(query, k=5):
    q = retrieval.embed_texts(model, processor, [query], device=device)[0]
    dists, idxs = retrieval.query_index(index, q, k=k)
    rows = [{"rank": i+1, "cluster": cluster_ids[int(idx)],
             "name": interpretations[cluster_ids.index(cluster_ids[int(idx)])]["name"],
             "similarity": round(float(d), 4)} for i, (d, idx) in enumerate(zip(dists[0], idxs[0]))]
    print("QUERY:", query)
    print(pd.DataFrame(rows).to_string(index=False))
    print()

for q in ["a cup of coffee", "skateboard tricks", "graffiti on a wall"]:
    show_top(q)

QUERY: a cup of coffee
 rank  cluster           name  similarity
    1       20     cup coffee      0.8392
    2       26      close eye      0.7448
    3       21 building clock      0.7322
    4       24     water tank      0.7272
    5       16  sitting chair      0.7200

QUERY: skateboard tricks
 rank  cluster             name  similarity
    1       13 skateboard doing      0.8852
    2       12          leo leo      0.7480
    3       21   building clock      0.7190
    4        7        bird long      0.7167
    5       16    sitting chair      0.7105

QUERY: graffiti on a wall
 rank  cluster           name  similarity
    1       15  graffiti wall      0.8017
    2       21 building clock      0.6811
    3       18    clock clock      0.6739
    4        9   rain covered      0.6629
    5       26      close eye      0.6560



## 4 · Retrieval evaluation (curated labels)

In [5]:
labels = json.loads((config.CLUSTER_METADATA_DIR / "retrieval_eval_labels.json").read_text())
labels = {k: v for k, v in labels.items() if isinstance(v, dict)}
print("queries:", len(labels))

results = retrieval.evaluate_retrieval(
    labels, retrieval.embed_texts, index, cluster_ids, k_values=(1, 3, 5),
    model=model, processor=processor, device=device,
)
pd.DataFrame(results["aggregate"], index=["value"]).T

queries: 20


,value
hit@1,0.8500
hit@3,0.8500
hit@5,0.9500
mrr,0.8725
n_queries,20.0000


In [6]:
df = pd.DataFrame(results["per_query"])
df[df["hit@1"] == 0][["query", "expected_clusters", "retrieved_top5"]]

,query,expected_clusters,retrieved_top5
0,a photo of a dog,[0],"[26, 1, 7, 0, 3]"
16,a baby,[22],"[26, 21, 3, 7, 18]"
17,dolls,"[19, 6]","[21, 3, 1, 18, 6]"


## 5 · Persist results

`retrieval_results.json` carries the integrity disclaimer + curation note alongside the metrics.

In [7]:
retrieval.save_eval_results(results)
print("saved -> artifacts/cluster_metadata/retrieval_results.json")

saved -> artifacts/cluster_metadata/retrieval_results.json


## Phase 6 checkpoint
- [x] CLIP text embeddings (same model as image side, cached)
- [x] FAISS flat IP index over clusters, persisted
- [x] hit@k / MRR against human-curated labels, persisted with disclaimer

**Next (Phase 7):** RAG for the frontend — retrieval + context assembly (`build_context`) for the React app, plus backend wiring (Stage 13).